# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Muneeb-th/ML-Assignment-1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [5]:
%pip -q install duckdb huggingface_hub

import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':        f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':        f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':         f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':  f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':     f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}
print("Connected.")

Connected.


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content item, for one client, on one day. This is the grain
of fact_content_daily_performance. Time window: I'll develop and verify
against month=2026-03 (a mid-panel month), and treat the final month
(the _sample table, June 2026) as a sealed test window I won't touch
until later.

In [6]:
grain_check = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS n
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
""").df()

print(f"Duplicate (client, content, day) combos found: {len(grain_check)}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate (client, content, day) combos found: 0


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features: gsc_impressions, gsc_clicks, gsc_avg_position — observable
before the decision point.
Label/proxy: I'll define is_declining as a month-over-month drop in
impressions greater than 20%, built only from data before the window I'm
predicting.
Context: client_hash_id, content_hash_id — used for joins/grouping only,
never as a feature.
Excluded: any product-computed flag (health_score, priority_score,
action_type). These aren't even shipped in this data, but I'm excluding
them on principle — feeding a product's own decision into a discovery
model just teaches it to copy that decision instead of finding real signal.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Verified below: row count and date span for month=2026-03, and
availability filtered on ga4_data_available.

In [8]:
overview = con.sql(f"""
    SELECT COUNT(*) AS row_count,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
""").df()
print("Row count + date span for month=2026-03:")
print(overview)

availability = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
""").df()
print("\nAvailability check:")
print(availability)

Row count + date span for month=2026-03:
   row_count   min_date   max_date
0    9841378 2026-03-01 2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Availability check:
   total_rows  ga4_available_rows
0     9841378            413966.0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This data can't tell you why a page moved — only that it moved. It's an
unbalanced panel: clients have different history lengths, so early rows
for newer clients may be GSC-only with ga4_data_available = FALSE, which
could look like "no traffic" when it's really "not tracked yet." My
feature and label windows must never overlap, or the result would be
leakage, not a real signal.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.